In [ ]:
"""
time_dependent_analysis.py
===========================
Time-dependent S21 resonator fitting and noise analysis.

Dataset layout
--------------
Each power subfolder (-20dBm, -40dBm, -60dBm, -80dBm) contains CSV files
that are individual S21 sweeps taken sequentially in time.

Timing model
------------
- One S21 sweep takes 54 s (measurement_time_s = 54)
- After one measurement the instrument waits 54*3 = 162 s before the next
  measurement OF THE SAME POWER.  In a 4-power interleaved scheme this means
  the full period between consecutive same-power points is:
       period = 54 (measure) + 54*3 (wait) = 216 s
  BUT the ordering is interleaved: the 4 powers cycle one after another,
  so the true elapsed time for point index i (0-based) of a given power is:
       t_i = i * N_powers * (measurement_time_s + wait_between_same) 
  ... simplified: we reconstruct the absolute time from the file sort order
  and the known per-point timing.

Noise analysis
--------------
Allan deviation, PSD, and drift fit are performed on each parameter series.
"""

# ── stdlib ──────────────────────────────────────────────────────────────────
import sys
import os
import re
import warnings
from pathlib import Path
import datetime

# ── third-party ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal, stats
import allantools          # pip install allantools

# ── fitting-code repo ────────────────────────────────────────────────────────
REPO_DIR = Path(__file__).resolve().parent / "Fitting_Code_Lab_2.0"
sys.path.insert(0, str(REPO_DIR / "helper_scripts"))
sys.path.insert(0, str(REPO_DIR / "scresonators"))

import helper_fit as hf
import helper_misc as hm
import fit_resonator.resonator as res
import fit_resonator.fit as fsd

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# USER CONFIGURATION  – edit these lines
# ════════════════════════════════════════════════════════════════════════════
DATA_ROOT = Path(
    r"C:\Users\user\Documents\GitHub\Measurements"
    r"\Cooldown_77_Line5-QSD_CPW_w6g3_03"
    r"\15mK_Resonator_2_5p608GHz_Time_Dependent_S21"
)
POWER_FOLDERS = ["-20dBm", "-40dBm", "-60dBm", "-80dBm"]

# Timing
MEASUREMENT_TIME_S = 54.0          # seconds to acquire one S21 sweep
WAIT_BETWEEN_SAME_POWER_PERIODS = 3  # number of measurement_times to wait between
                                      # consecutive same-power measurements
N_POWERS = len(POWER_FOLDERS)       # 4 power levels interleaved

# The period for ONE power is: all N_POWERS measurements + the wait
# Period = N_POWERS * MEASUREMENT_TIME_S + WAIT * MEASUREMENT_TIME_S
# But from the problem statement: "take 54 s per point, wait 54*3, spend 54 s"
# meaning the inter-sample interval for ONE power is (1 + 3)*54 = 4*54 = 216 s
PERIOD_S = MEASUREMENT_TIME_S * (1 + WAIT_BETWEEN_SAME_POWER_PERIODS)

# Fitting
PREPROCESS_METHOD = "circle"
MC_ITERATION = 10
MC_ROUNDS    = 10
PHI0 = 0.0

# Output directory (next to this script)
OUT_DIR = Path(__file__).resolve().parent / "time_dep_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════════
# HELPER UTILITIES
# ════════════════════════════════════════════════════════════════════════════

def sorted_csv_files(folder: Path):
    """Return CSV files sorted by name (assumes chronological naming)."""
    csvs = sorted(folder.glob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSV files found in {folder}")
    return csvs


def fit_one_sweep(csv_path: Path, save_dir: Path):
    """
    Run DCM circle fit on a single S21 CSV file.

    Returns
    -------
    dict with keys: Q, Qi, Qc, phi, fc, Q_err, Qi_err, Qc_err, phi_err, fc_err
    or None on failure.
    """
    try:
        myres = res.Resonator()
        myres.from_file(str(csv_path))
        myres.preprocess_method = PREPROCESS_METHOD
        myres.normalize = 5
        myres.save_dcm_plot = False
        myres.plot_extra = False
        myres.plot = "png"
        myres.fit_dir = str(save_dir)

        myres.fit_method(
            "DCM",
            MC_ITERATION,
            MC_rounds=MC_ROUNDS,
            MC_fix=[],
            manual_init=None,
            MC_step_const=0.3,
        )

        fit_result = fsd.fit(myres)
        if fit_result is None:
            print(f"  [WARN] fit returned None for {csv_path.name}")
            return None

        params, conf_intervals, err, init1, fig = fit_result
        plt.close("all")

        # params = [Q, Qc, fc, phi, ...]   (DCM convention in this repo)
        Q   = params[0]
        Qc_raw = params[1]
        fc  = params[2]
        phi = params[3]

        Qcj = Qc_raw * np.exp(1j * (phi + PHI0))
        Qi  = float(np.real(1.0 / (1.0 / Q - np.real(1.0 / Qcj))))
        Qc  = float(np.real(Qcj))

        return {
            "Q":      float(Q),
            "Qi":     Qi,
            "Qc":     Qc,
            "phi":    float(phi),
            "fc":     float(fc),
            "Q_err":  float(conf_intervals[0]),
            "Qi_err": float(conf_intervals[1]),
            "Qc_err": float(conf_intervals[2]),
            "phi_err":float(conf_intervals[3]) if len(conf_intervals) > 3 else np.nan,
            "fc_err": float(conf_intervals[5]) if len(conf_intervals) > 5 else np.nan,
        }

    except Exception as exc:
        print(f"  [ERROR] {csv_path.name}: {exc}")
        return None


# ════════════════════════════════════════════════════════════════════════════
# STEP 1 – FIT ALL SWEEPS FOR EACH POWER
# ════════════════════════════════════════════════════════════════════════════

def fit_all_powers(data_root: Path, power_folders: list[str]):
    """
    Iterate over each power subfolder, fit every CSV, build a DataFrame.

    Returns
    -------
    dict[str, pd.DataFrame]  keyed by power string e.g. "-20dBm"
    """
    all_results = {}

    for pwr_name in power_folders:
        folder = data_root / pwr_name
        if not folder.exists():
            print(f"[SKIP] folder not found: {folder}")
            continue

        print(f"\n{'='*60}")
        print(f"Processing {pwr_name}  ({folder})")
        print(f"{'='*60}")

        csvs = sorted_csv_files(folder)
        print(f"  Found {len(csvs)} CSV files")

        save_dir = OUT_DIR / pwr_name / "circle_fits"
        save_dir.mkdir(parents=True, exist_ok=True)

        rows = []
        for idx, csv_path in enumerate(csvs):
            print(f"  [{idx+1:03d}/{len(csvs)}] {csv_path.name}")
            result = fit_one_sweep(csv_path, save_dir)

            if result is not None:
                result["file"]    = csv_path.name
                result["index"]   = idx
                # Build absolute time axis:
                # Point i of THIS power was measured at t = i * PERIOD_S
                result["time_s"]  = idx * PERIOD_S
                result["time_min"] = result["time_s"] / 60.0
                result["time_hr"]  = result["time_s"] / 3600.0
                rows.append(result)
            else:
                rows.append({"file": csv_path.name, "index": idx,
                             "time_s": idx * PERIOD_S,
                             "time_min": idx * PERIOD_S / 60.0,
                             "time_hr": idx * PERIOD_S / 3600.0})

        df = pd.DataFrame(rows)
        df = df.sort_values("index").reset_index(drop=True)
        all_results[pwr_name] = df

        # Save CSV
        csv_out = OUT_DIR / pwr_name / "fit_results.csv"
        df.to_csv(csv_out, index=False)
        print(f"  Saved → {csv_out}")

    return all_results


# ════════════════════════════════════════════════════════════════════════════
# STEP 2 – PLOT PARAMETERS VERSUS TIME (with uncertainties)
# ════════════════════════════════════════════════════════════════════════════

PARAM_LABELS = {
    "Q":   (r"$Q$",              ""),
    "Qi":  (r"$Q_i$",           ""),
    "Qc":  (r"$Q_c$",           ""),
    "phi": (r"$\phi$",          " [rad]"),
    "fc":  (r"$f_c$",           " [Hz]"),
}
COLORS = plt.rcParams["axes.prop_cycle"].by_key()["color"]


def plot_params_vs_time(all_results: dict[str, pd.DataFrame]):
    """
    One figure per parameter, all powers overlaid, with error bars.
    """
    params = ["Q", "Qi", "Qc", "phi", "fc"]

    for param in params:
        label, unit = PARAM_LABELS[param]
        err_col = param + "_err"

        fig, ax = plt.subplots(figsize=(10, 5))

        for ci, (pwr_name, df) in enumerate(all_results.items()):
            if param not in df.columns:
                continue
            mask = df[param].notna() & df[err_col].notna() if err_col in df.columns else df[param].notna()
            t   = df.loc[mask, "time_hr"].values
            val = df.loc[mask, param].values
            err = df.loc[mask, err_col].values if err_col in df.columns and mask.any() else None

            ax.errorbar(
                t, val, yerr=err,
                fmt="o-", ms=4, lw=1, capsize=3,
                color=COLORS[ci % len(COLORS)],
                label=pwr_name,
                alpha=0.85,
            )

        ax.set_xlabel("Time [hours]", fontsize=13)
        ax.set_ylabel(label + unit, fontsize=13)
        ax.set_title(f"{label} vs Time — all powers", fontsize=14)
        ax.legend(fontsize=10)
        ax.grid(True, ls="--", alpha=0.4)
        fig.tight_layout()

        fig_path = OUT_DIR / f"{param}_vs_time.png"
        fig.savefig(fig_path, dpi=150)
        plt.close(fig)
        print(f"  Saved → {fig_path}")

    # Combined panel figure
    fig, axes = plt.subplots(len(params), 1, figsize=(12, 4 * len(params)), sharex=True)

    for pi, param in enumerate(params):
        ax = axes[pi]
        label, unit = PARAM_LABELS[param]
        err_col = param + "_err"

        for ci, (pwr_name, df) in enumerate(all_results.items()):
            if param not in df.columns:
                continue
            mask = df[param].notna()
            t   = df.loc[mask, "time_hr"].values
            val = df.loc[mask, param].values
            err = df.loc[mask, err_col].values if err_col in df.columns and mask.any() else None

            ax.errorbar(t, val, yerr=err, fmt="o-", ms=3, lw=0.8,
                        capsize=2, color=COLORS[ci % len(COLORS)],
                        label=pwr_name, alpha=0.85)

        ax.set_ylabel(label + unit, fontsize=11)
        ax.grid(True, ls="--", alpha=0.3)
        if pi == 0:
            ax.legend(fontsize=9, ncol=2)

    axes[-1].set_xlabel("Time [hours]", fontsize=13)
    fig.suptitle("Resonator Parameters vs Time", fontsize=15, y=1.002)
    fig.tight_layout()
    fig_path = OUT_DIR / "all_params_vs_time_panel.png"
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {fig_path}")


# ════════════════════════════════════════════════════════════════════════════
# STEP 3 – NOISE ANALYSIS
# ════════════════════════════════════════════════════════════════════════════

def compute_adev(series: np.ndarray, tau0: float):
    """
    Compute overlapping Allan deviation using allantools.

    Parameters
    ----------
    series  : 1-D array of parameter values (evenly spaced)
    tau0    : sample period in seconds

    Returns
    -------
    taus, adev, adeverr, n
    """
    # allantools wants fractional frequency; we normalise by the mean
    mean_val = np.nanmean(series)
    if mean_val == 0 or not np.isfinite(mean_val):
        return None, None, None, None
    frac = (series - mean_val) / mean_val

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        taus, adev, adeverr, n = allantools.oadev(
            frac, rate=1.0 / tau0, data_type="freq", taus="all"
        )
    return taus, adev, adeverr, n


def compute_psd(series: np.ndarray, fs: float):
    """
    Welch PSD of the de-trended series.
    Returns freqs [Hz], psd [units²/Hz].
    """
    detrended = signal.detrend(series[np.isfinite(series)])
    nperseg = min(len(detrended), 64)
    freqs, psd = signal.welch(detrended, fs=fs, nperseg=nperseg,
                              scaling="density")
    return freqs, psd


def noise_analysis(all_results: dict[str, pd.DataFrame]):
    """
    For each power and each parameter:
      1. Allan deviation (overlapping)
      2. Welch PSD
      3. Linear drift fit
    Saves summary plots and a CSV.
    """
    params = ["Q", "Qi", "Qc", "phi", "fc"]
    fs = 1.0 / PERIOD_S           # sample rate [Hz]
    tau0 = PERIOD_S               # sample interval [s]

    noise_dir = OUT_DIR / "noise_analysis"
    noise_dir.mkdir(parents=True, exist_ok=True)

    summary_rows = []

    for pwr_name, df in all_results.items():
        pwr_dir = noise_dir / pwr_name
        pwr_dir.mkdir(parents=True, exist_ok=True)

        for param in params:
            if param not in df.columns:
                continue

            series = df[param].values.astype(float)
            t_hr   = df["time_hr"].values.astype(float)
            valid  = np.isfinite(series)

            if valid.sum() < 5:
                print(f"  [SKIP noise] {pwr_name} / {param}: too few valid points ({valid.sum()})")
                continue

            s_valid = series[valid]
            t_valid = t_hr[valid]
            mean_val = np.nanmean(s_valid)
            std_val  = np.nanstd(s_valid)

            # ── Linear drift fit ─────────────────────────────────────────
            slope, intercept, r_val, p_val, slope_stderr = stats.linregress(t_valid, s_valid)
            drift_per_hr = slope
            drift_frac   = slope / mean_val if mean_val != 0 else np.nan

            # ── Allan deviation ──────────────────────────────────────────
            taus, adev, adeverr, _ = compute_adev(s_valid, tau0)

            # ── PSD ──────────────────────────────────────────────────────
            freqs_psd, psd = compute_psd(s_valid, fs)

            # ── FIGURE ───────────────────────────────────────────────────
            fig = plt.figure(figsize=(15, 10))
            gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.4)

            label, unit = PARAM_LABELS[param]

            # Panel 1: time series + drift line
            ax_ts = fig.add_subplot(gs[0, :2])
            ax_ts.plot(t_valid, s_valid, "o-", ms=3, lw=0.8, alpha=0.8, label="data")
            t_fit = np.linspace(t_valid.min(), t_valid.max(), 200)
            ax_ts.plot(t_fit, slope * t_fit + intercept, "r--", lw=1.5,
                       label=f"drift {drift_per_hr:+.3g} {unit}/hr")
            ax_ts.set_xlabel("Time [hr]")
            ax_ts.set_ylabel(label + unit)
            ax_ts.set_title(f"{pwr_name}  |  {label}  time series")
            ax_ts.legend(fontsize=9)
            ax_ts.grid(True, ls="--", alpha=0.3)

            # Panel 2: histogram
            ax_hist = fig.add_subplot(gs[0, 2])
            ax_hist.hist(s_valid, bins="auto", edgecolor="k", alpha=0.7)
            ax_hist.axvline(mean_val, color="r", lw=1.5, label=f"mean={mean_val:.4g}")
            ax_hist.set_xlabel(label + unit)
            ax_hist.set_ylabel("Counts")
            ax_hist.set_title("Distribution")
            ax_hist.legend(fontsize=8)

            # Panel 3: Allan deviation
            ax_ad = fig.add_subplot(gs[1, 0])
            if taus is not None and len(taus) > 1:
                ax_ad.loglog(taus, adev, "b.-", ms=5)
                ax_ad.fill_between(taus,
                                   np.array(adev) - np.array(adeverr),
                                   np.array(adev) + np.array(adeverr),
                                   alpha=0.25, color="b")
            ax_ad.set_xlabel("Averaging time τ [s]")
            ax_ad.set_ylabel(f"OADEV of Δ{label}/{label}")
            ax_ad.set_title("Overlapping Allan Deviation")
            ax_ad.grid(True, which="both", ls="--", alpha=0.3)

            # Panel 4: PSD
            ax_psd = fig.add_subplot(gs[1, 1])
            if len(freqs_psd) > 1:
                ax_psd.semilogy(freqs_psd[1:], psd[1:], color="g")
            ax_psd.set_xlabel("Frequency [Hz]")
            ax_psd.set_ylabel(f"PSD [{unit if unit else 'a.u.'}²/Hz]")
            ax_psd.set_title("Welch PSD (de-trended)")
            ax_psd.grid(True, ls="--", alpha=0.3)

            # Panel 5: de-trended residuals
            ax_res = fig.add_subplot(gs[1, 2])
            residuals = s_valid - (slope * t_valid + intercept)
            ax_res.plot(t_valid, residuals, "k-", lw=0.8, alpha=0.8)
            ax_res.axhline(0, color="r", lw=1)
            ax_res.set_xlabel("Time [hr]")
            ax_res.set_ylabel(f"Δ{label} (residual)")
            ax_res.set_title(f"De-trended  RMS={np.std(residuals):.3g}")
            ax_res.grid(True, ls="--", alpha=0.3)

            fig.suptitle(
                f"Noise Analysis | {pwr_name} | {label}\n"
                f"mean={mean_val:.5g}  std={std_val:.3g}  drift={drift_per_hr:+.3g}/hr  "
                f"R²={r_val**2:.3f}",
                fontsize=12, y=1.01
            )

            fig_path = pwr_dir / f"noise_{param}.png"
            fig.savefig(fig_path, dpi=150, bbox_inches="tight")
            plt.close(fig)
            print(f"  Saved → {fig_path}")

            # Minimum Allan dev and its tau
            if taus is not None and len(adev) > 0:
                amin_idx = np.argmin(adev)
                adev_min = adev[amin_idx]
                adev_min_tau = taus[amin_idx]
            else:
                adev_min = np.nan
                adev_min_tau = np.nan

            summary_rows.append({
                "power":         pwr_name,
                "parameter":     param,
                "mean":          mean_val,
                "std":           std_val,
                "relative_std":  std_val / abs(mean_val) if mean_val != 0 else np.nan,
                "drift_per_hr":  drift_per_hr,
                "drift_frac_per_hr": drift_frac,
                "R2":            r_val**2,
                "adev_min":      adev_min,
                "adev_min_tau_s":adev_min_tau,
                "n_points":      valid.sum(),
            })

    summary_df = pd.DataFrame(summary_rows)
    summary_path = noise_dir / "noise_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"\n  Noise summary → {summary_path}")

    # ── Cross-power Allan deviation comparison ───────────────────────────
    for param in params:
        fig, ax = plt.subplots(figsize=(8, 5))
        label, unit = PARAM_LABELS[param]

        for ci, (pwr_name, df) in enumerate(all_results.items()):
            if param not in df.columns:
                continue
            series = df[param].values.astype(float)
            valid  = np.isfinite(series)
            if valid.sum() < 5:
                continue
            taus, adev, adeverr, _ = compute_adev(series[valid], tau0)
            if taus is None:
                continue
            ax.loglog(taus, adev, ".-", ms=5, lw=1, color=COLORS[ci % len(COLORS)],
                      label=pwr_name)
            ax.fill_between(taus,
                            np.array(adev) - np.array(adeverr),
                            np.array(adev) + np.array(adeverr),
                            alpha=0.15, color=COLORS[ci % len(COLORS)])

        ax.set_xlabel("Averaging time τ [s]", fontsize=12)
        ax.set_ylabel(f"OADEV of Δ{label}/{label}", fontsize=12)
        ax.set_title(f"Allan Deviation — {label} — all powers", fontsize=13)
        ax.legend(fontsize=10)
        ax.grid(True, which="both", ls="--", alpha=0.3)
        fig.tight_layout()

        fig_path = noise_dir / f"adev_comparison_{param}.png"
        fig.savefig(fig_path, dpi=150)
        plt.close(fig)
        print(f"  Saved → {fig_path}")

    return summary_df


# ════════════════════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════════════════════

def main():
    print("=" * 70)
    print("Time-Dependent Resonator Analysis")
    print(f"  Data root : {DATA_ROOT}")
    print(f"  Tau0      : {PERIOD_S} s  ({PERIOD_S/60:.2f} min)")
    print(f"  fs        : {1/PERIOD_S*1e3:.3f} mHz")
    print("=" * 70)

    # ── Install allantools if needed ──────────────────────────────────────
    try:
        import allantools
    except ImportError:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "allantools", "--break-system-packages", "-q"])
        import allantools

    # ── Step 1: Fit all sweeps ─────────────────────────────────────────────
    print("\n[STEP 1] Circle fitting all sweeps …")
    all_results = fit_all_powers(DATA_ROOT, POWER_FOLDERS)

    if not all_results:
        print("[ERROR] No data found. Check DATA_ROOT.")
        return

    # ── Step 2: Plot parameters vs time ───────────────────────────────────
    print("\n[STEP 2] Plotting parameters vs time …")
    plot_params_vs_time(all_results)

    # ── Step 3: Noise analysis ─────────────────────────────────────────────
    print("\n[STEP 3] Noise analysis …")
    summary = noise_analysis(all_results)

    print("\n" + "=" * 70)
    print("NOISE SUMMARY")
    print("=" * 70)
    if not summary.empty:
        print(summary.to_string(index=False))

    print(f"\nAll outputs saved to: {OUT_DIR}")
    print("Done.")


if __name__ == "__main__":
    main()